## Web Scraping - Bank of Abyssinia

In [34]:
# The unique identifier for Bank of Abyssinia's app on the Google Play Store
ABYSSINIA_ID = 'com.boa.boaMobileBanking'

# Step 1: Get app metadata (rating, installs, description...)
abyssinia_app_info = app(
    ABYSSINIA_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("=" * 50)
print("Bank of Abyssinia App Info")
print("=" * 50)
print(f"App Title   : {abyssinia_app_info['title']}")
print(f"Current Score: {abyssinia_app_info['score']}")
print(f"Total Ratings: {abyssinia_app_info['ratings']:,}")
print(f"Total Reviews: {abyssinia_app_info['reviews']:,}")
print(f"Installs     : {abyssinia_app_info['installs']}")

Bank of Abyssinia App Info
App Title   : BoA Mobile
Current Score: 4.3922596
Total Ratings: 9,219
Total Reviews: 1,461
Installs     : 1,000,000+


In [35]:
# Step 2: Scrape reviews for Bank of Abyssinia
print(f"Scraping reviews for Bank of Abyssinia...")

abyssinia_result, abyssinia_continuation_token = reviews(
    ABYSSINIA_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       # Most recent first
    count=400,              # Ask for more than 400 to be safe
    filter_score_with=None  # All star ratings
)

print(f"Collected {len(abyssinia_result)} raw reviews for Bank of Abyssinia")

Scraping reviews for Bank of Abyssinia...
Collected 400 raw reviews for Bank of Abyssinia


In [36]:
# Step 3: Extract required fields from reviews for Bank of Abyssinia
abyssinia_reviews_data = []

for review in abyssinia_result:
    abyssinia_reviews_data.append({
        'review_id': review.get('reviewId', ''),
        'review_text': review.get('reviewText', review.get('content', '')),
        'rating': review.get('score', 0),
        'review_date': review.get('at', ''),
        'bank': 'Bank of Abyssinia',
        'app_name': 'BANK OF ABYSSINIA',
        'source': 'Google Play'
    })

abyssinia_reviews = pd.DataFrame(abyssinia_reviews_data)
print(f"\nRaw reviews before deduplication for Bank of Abyssinia: {len(abyssinia_reviews)}")
abyssinia_reviews = abyssinia_reviews.drop_duplicates(subset=['review_id'], keep='first').reset_index(drop=True)
print(f"Reviews after deduplication by review_id for Bank of Abyssinia: {len(abyssinia_reviews)}")

print("\n" + "=" * 50)
print("Bank of Abyssinia Reviews DataFrame")
print("=" * 50)
print(f"Total reviews collected: {len(abyssinia_reviews)}")
print(f"\nColumns: {list(abyssinia_reviews.columns)}")
print(f"\nFirst 3 reviews:")
print(abyssinia_reviews[['bank', 'rating', 'review_date', 'source']].head(3))
print(f"\nRating distribution:")
print(abyssinia_reviews['rating'].value_counts().sort_index())


Raw reviews before deduplication for Bank of Abyssinia: 400
Reviews after deduplication by review_id for Bank of Abyssinia: 400

Bank of Abyssinia Reviews DataFrame
Total reviews collected: 400

Columns: ['review_id', 'review_text', 'rating', 'review_date', 'bank', 'app_name', 'source']

First 3 reviews:
                bank  rating         review_date       source
0  Bank of Abyssinia       5 2026-05-12 08:50:32  Google Play
1  Bank of Abyssinia       2 2026-05-11 15:18:54  Google Play
2  Bank of Abyssinia       5 2026-05-09 11:41:50  Google Play

Rating distribution:
rating
1    112
2     13
3     13
4     32
5    230
Name: count, dtype: int64


## Preprocessing - Bank of Abyssinia

In [37]:
# View first 10 rows of the required fields only for Bank of Abyssinia
required_columns = ['review_text', 'rating', 'review_date', 'bank', 'app_name', 'source', 'review_id']

# Reorder the DataFrame to ensure the columns appear in the requested order
abyssinia_reviews = abyssinia_reviews[required_columns]

print("Required columns:")
print(abyssinia_reviews.columns.tolist())
print("\nFirst 10 reviews with the required fields:")
print(abyssinia_reviews.head(10))

print("\nReview text availability:")
print(f"Non-empty review_text rows: {abyssinia_reviews['review_text'].astype(bool).sum()}")
print(f"Empty review_text rows: {len(abyssinia_reviews) - abyssinia_reviews['review_text'].astype(bool).sum()}")

Required columns:
['review_text', 'rating', 'review_date', 'bank', 'app_name', 'source', 'review_id']

First 10 reviews with the required fields:
                                         review_text  rating  \
0                                 it's very good app       5   
1  this app is good but the speed of app is very ...       2   
2                                               good       5   
3                                       boa the best       5   
4         bank of absiniya is best bank in ethiopian       5   
5                                   አስተማማኝና ዘመኑን የዋጀ       4   
6                                               good       5   
7  extremely slow app and unreliable for most pay...       2   
8                                        Amazing app       5   
9  I tried to oppen mobile app of BOA, but it can...       1   

          review_date               bank           app_name       source  \
0 2026-05-12 08:50:32  Bank of Abyssinia  BANK OF ABYSSINIA  Google Play 

In [38]:
# Handle missing values for Bank of Abyssinia
abyssinia_reviews = abyssinia_reviews.dropna(subset=['review_text', 'rating'])
print(f"Total reviews after dropping missing values for Bank of Abyssinia: {len(abyssinia_reviews)}")
print(f"Final dataset shape for Bank of Abyssinia: {abyssinia_reviews.shape}")

Total reviews after dropping missing values for Bank of Abyssinia: 400
Final dataset shape for Bank of Abyssinia: (400, 7)


In [39]:
# Normalize dates to YYYY-MM-DD format for Bank of Abyssinia
abyssinia_reviews['review_date'] = pd.to_datetime(abyssinia_reviews['review_date'], errors='coerce').dt.strftime('%Y-%m-%d')
print(f"Final dataset shape for Bank of Abyssinia: {abyssinia_reviews.shape}")

Final dataset shape for Bank of Abyssinia: (400, 7)


In [40]:
# Get the date range for Bank of Abyssinia
abyssinia_start_date = abyssinia_reviews['review_date'].min()
abyssinia_end_date = abyssinia_reviews['review_date'].max()

print(f"Bank of Abyssinia data spans from: {abyssinia_start_date}")
print(f"Up to: {abyssinia_end_date}")

Bank of Abyssinia data spans from: 2025-06-13
Up to: 2026-05-12


In [41]:
# Save the cleaned Bank of Abyssinia dataset as a CSV
abyssinia_reviews.to_csv('abyssinia_reviews_cleaned.csv', index=False)

In [42]:
from google.colab import files

files.download('abyssinia_reviews_cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Data Quality Summary for Cleaned Reviews

### CBE Bank Reviews (`cbe_reviews_cleaned.csv`)

*   **Total Reviews:** The dataset contains `450` reviews.
*   **Columns:** 7 columns: `review_text`, `rating`, `review_date`, `bank`, `app_name`, `source`, `review_id`.
*   **Missing Values:** All missing `review_text` and `rating` values were handled by dropping corresponding rows. The final dataset has no missing values in these critical columns.
*   **Date Range:** Reviews span from `2026-03-10` to `2026-05-14`.
*   **Duplicates:** Duplicate reviews based on `review_id` were removed (though none were found in the initial scrape).
*   **Review Text Availability:** All `450` reviews have non-empty `review_text`.

### Dashen Bank Reviews (`dashen_reviews_cleaned.csv`)

*   **Total Reviews:** The dataset contains `400` reviews.
*   **Columns:** 7 columns: `review_text`, `rating`, `review_date`, `bank`, `app_name`, `source`, `review_id`.
*   **Missing Values:** All missing `review_text` and `rating` values were handled by dropping corresponding rows. The final dataset has no missing values in these critical columns.
*   **Date Range:** Reviews span from `2025-09-28` to `2026-05-13`.
*   **Duplicates:** Duplicate reviews based on `review_id` were removed (though none were found in the initial scrape).
*   **Review Text Availability:** All `400` reviews have non-empty `review_text`.

### Bank of Abyssinia Reviews (`abyssinia_reviews_cleaned.csv`)

*   **Total Reviews:** The dataset contains `400` reviews.
*   **Columns:** 7 columns: `review_text`, `rating`, `review_date`, `bank`, `app_name`, `source`, `review_id`.
*   **Missing Values:** All missing `review_text` and `rating` values were handled by dropping corresponding rows. The final dataset has no missing values in these critical columns.
*   **Date Range:** Reviews span from `2025-06-13` to `2026-05-12`.
*   **Duplicates:** Duplicate reviews based on `review_id` were removed (though none were found in the initial scrape).
*   **Review Text Availability:** All `400` reviews have non-empty `review_text`.

All three datasets are now cleaned and structured with consistent column names and formats, ready for further analysis.